# IPLACEX

## Delta Calculous

In [ ]:
import pandas as pd

# Try reading the CSV file and check the columns
df = pd.read_csv('data/data-iplacex.csv', sep=',', on_bad_lines='skip', header=0, encoding='utf-8-sig')

# Check if 'session_id' is now a column
if 'session_id' not in df.columns:
    print("Column 'session_id' not found. Columns are:", df.columns)
else:
    # First, sort the DataFrame by session_id and timestamp
    df_sorted = df.sort_values(['session_id', 'timestamp'])

    # Convert timestamp to datetime
    df_sorted['timestamp'] = pd.to_datetime(df_sorted['timestamp'], format='%Y-%m-%dT%H:%M:%S.%f')

    # Calculate delta only between messages of the same session
    df_sorted['delta'] = df_sorted['timestamp'].diff().dt.total_seconds() / 60

	# Set delta to 0 when it's a new session
    df_sorted['delta'] = df_sorted.apply(lambda row:
        0 if pd.isna(row['delta']) or df_sorted['session_id'].shift(1).loc[row.name] != row['session_id']
        else row['delta'], axis=1)

    # Round to 2 decimal places
    df_sorted['delta'] = df_sorted['delta'].round(2)

    # Save the result to a new CSV file
    df_sorted.to_csv('data/chats.csv', index=False, sep=',', encoding='utf-8-sig')